In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path


In [2]:
DATA_PATH = Path("../data/fake_job_postings.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (17880, 18)
Columns:
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

In [4]:
print(df.shape)
print(df.columns.tolist())

(17880, 18)
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


In [5]:
# Create presence / missing information features

text_columns = [
    "department",
    "salary_range",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

for col in text_columns:
    df[f"has_{col}"] = df[col].notna().astype(int)

print("New shape:", df.shape)

presence_features = [f"has_{col}" for col in text_columns]

print("\nPresence features:")
print(presence_features)

df[presence_features].head()

New shape: (17880, 29)

Presence features:
['has_department', 'has_salary_range', 'has_company_profile', 'has_description', 'has_requirements', 'has_benefits', 'has_employment_type', 'has_required_experience', 'has_required_education', 'has_industry', 'has_function']


,has_department,has_salary_range,has_company_profile,has_description,has_requirements,has_benefits,has_employment_type,has_required_experience,has_required_education,has_industry,has_function
0,1,0,1,1,1,0,1,1,0,0,1
1,1,0,1,1,1,1,1,1,0,1,1
2,0,0,1,1,1,0,0,0,0,0,0
3,1,0,1,1,1,1,1,1,1,1,1
4,0,0,1,1,1,1,1,1,1,1,1


In [6]:
print(df[presence_features].sum())

has_department              6333
has_salary_range            2868
has_company_profile        14572
has_description            17879
has_requirements           15184
has_benefits               10668
has_employment_type        14409
has_required_experience    10830
has_required_education      9775
has_industry               12977
has_function               11425
dtype: int64


In [7]:
print(df[presence_features].mean().round(3))

has_department             0.354
has_salary_range           0.160
has_company_profile        0.815
has_description            1.000
has_requirements           0.849
has_benefits               0.597
has_employment_type        0.806
has_required_experience    0.606
has_required_education     0.547
has_industry               0.726
has_function               0.639
dtype: float64


In [8]:
print(df.shape)

(17880, 29)


In [9]:
text_length_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

In [10]:
for col in text_length_columns:
    df[f"{col}_char_length"] = df[col].fillna("").astype(str).str.len()
    df[f"{col}_word_count"] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

In [11]:
length_features = []

for col in text_length_columns:
    length_features.extend([
        f"{col}_char_length",
        f"{col}_word_count"
    ])

print("Created features:")
print(length_features)

print("\nNew shape:", df.shape)

Created features:
['title_char_length', 'title_word_count', 'company_profile_char_length', 'company_profile_word_count', 'description_char_length', 'description_word_count', 'requirements_char_length', 'requirements_word_count', 'benefits_char_length', 'benefits_word_count']

New shape: (17880, 39)


In [12]:
df[length_features].describe().T

,count,mean,std,min,25%,50%,75%,max
title_char_length,17880.0,28.530313,13.871256,3.0,19.0,25.0,35.0,142.0
title_word_count,17880.0,3.761913,2.074126,1.0,2.0,3.0,5.0,19.0
company_profile_char_length,17880.0,620.901454,567.454100,0.0,138.0,570.0,879.0,6178.0
company_profile_word_count,17880.0,92.553579,85.487278,0.0,21.0,83.0,132.0,885.0
description_char_length,17880.0,1218.004418,894.828620,0.0,607.0,1017.0,1586.0,14907.0
description_word_count,17880.0,170.445973,123.304236,0.0,87.0,146.0,224.0,2115.0
requirements_char_length,17880.0,590.132215,613.191270,0.0,146.0,467.0,820.0,10864.0
requirements_word_count,17880.0,78.032942,81.652956,0.0,19.0,62.0,108.0,1645.0
benefits_char_length,17880.0,208.895694,337.077357,0.0,0.0,45.0,294.0,4429.0
benefits_word_count,17880.0,29.990884,49.755790,0.0,0.0,6.0,41.0,694.0


In [13]:
import re
import numpy as np

def parse_salary_range(value):
    if pd.isna(value):
        return np.nan, np.nan, 0

    text = str(value).strip()

    # Extract numeric salary ranges such as:
    # 20000-28000
    # 100000-120000
    match = re.search(
        r'(\d+(?:,\d{3})*(?:\.\d+)?)\s*[-–]\s*(\d+(?:,\d{3})*(?:\.\d+)?)',
        text
    )

    if match:
        salary_min = float(match.group(1).replace(",", ""))
        salary_max = float(match.group(2).replace(",", ""))

        return salary_min, salary_max, 1

    return np.nan, np.nan, 0

In [14]:
salary_features = df["salary_range"].apply(parse_salary_range)

df["salary_min"] = salary_features.apply(lambda x: x[0])
df["salary_max"] = salary_features.apply(lambda x: x[1])
df["salary_is_range"] = salary_features.apply(lambda x: x[2])

print(df[[
    "salary_range",
    "salary_min",
    "salary_max",
    "salary_is_range"
]].head(20))

     salary_range  salary_min  salary_max  salary_is_range
0             NaN         NaN         NaN                0
1             NaN         NaN         NaN                0
2             NaN         NaN         NaN                0
3             NaN         NaN         NaN                0
4             NaN         NaN         NaN                0
5             NaN         NaN         NaN                0
6     20000-28000     20000.0     28000.0                1
7             NaN         NaN         NaN                0
8             NaN         NaN         NaN                0
9             NaN         NaN         NaN                0
10  100000-120000    100000.0    120000.0                1
11            NaN         NaN         NaN                0
12            NaN         NaN         NaN                0
13            NaN         NaN         NaN                0
14            NaN         NaN         NaN                0
15  120000-150000    120000.0    150000.0               

In [15]:
print("New shape:", df.shape)

print("\nSalary range count:")
print(df["salary_is_range"].value_counts())

print("\nParsed salary examples:")
print(
    df.loc[
        df["salary_is_range"] == 1,
        ["salary_range", "salary_min", "salary_max"]
    ].head(10)
)

New shape: (17880, 42)

Salary range count:
salary_is_range
0    15039
1     2841
Name: count, dtype: int64

Parsed salary examples:
     salary_range  salary_min  salary_max
6     20000-28000     20000.0     28000.0
10  100000-120000    100000.0    120000.0
15  120000-150000    120000.0    150000.0
23  100000-120000    100000.0    120000.0
31    50000-65000     50000.0     65000.0
42    40000-50000     40000.0     50000.0
65          60-80        60.0        80.0
76    65000-70000     65000.0     70000.0
77         75-115        75.0       115.0
79   75000-110000     75000.0    110000.0


In [16]:
df["salary_midpoint"] = (
    df["salary_min"] + df["salary_max"]
) / 2

df["salary_range_width"] = (
    df["salary_max"] - df["salary_min"]
)

print(df[[
    "salary_range",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_range_width"
]].loc[df["salary_is_range"] == 1].head(10))

     salary_range  salary_min  salary_max  salary_midpoint  salary_range_width
6     20000-28000     20000.0     28000.0          24000.0              8000.0
10  100000-120000    100000.0    120000.0         110000.0             20000.0
15  120000-150000    120000.0    150000.0         135000.0             30000.0
23  100000-120000    100000.0    120000.0         110000.0             20000.0
31    50000-65000     50000.0     65000.0          57500.0             15000.0
42    40000-50000     40000.0     50000.0          45000.0             10000.0
65          60-80        60.0        80.0             70.0                20.0
76    65000-70000     65000.0     70000.0          67500.0              5000.0
77         75-115        75.0       115.0             95.0                40.0
79   75000-110000     75000.0    110000.0          92500.0             35000.0


In [17]:
print(df.shape)

(17880, 44)


In [18]:
def create_location_features(value):
    if pd.isna(value):
        return 0, 0, 0

    text = str(value).strip().lower()

    has_location = int(bool(text))

    remote_pattern = r"\bremote\b|\bwork\s+from\s+home\b|\bwork\s+from\s+anywhere\b"
    hybrid_pattern = r"\bhybrid\b"

    is_remote = int(bool(re.search(remote_pattern, text)))
    is_hybrid = int(bool(re.search(hybrid_pattern, text)))

    return has_location, is_remote, is_hybrid


location_features = df["location"].apply(create_location_features)

df["has_location"] = location_features.apply(lambda x: x[0])
df["is_remote"] = location_features.apply(lambda x: x[1])
df["is_hybrid"] = location_features.apply(lambda x: x[2])

print(df[[
    "location",
    "has_location",
    "is_remote",
    "is_hybrid"
]].head(20))

                    location  has_location  is_remote  is_hybrid
0           US, NY, New York             1          0          0
1             NZ, , Auckland             1          0          0
2              US, IA, Wever             1          0          0
3         US, DC, Washington             1          0          0
4         US, FL, Fort Worth             1          0          0
5                   US, MD,              1          0          0
6             DE, BE, Berlin             1          0          0
7      US, CA, San Francisco             1          0          0
8          US, FL, Pensacola             1          0          0
9            US, AZ, Phoenix             1          0          0
10       US, NJ, Jersey City             1          0          0
11           GB, LND, London             1          0          0
12          US, CT, Stamford             1          0          0
13           US, FL, Orlando             1          0          0
14           AU, NSW, Syd

In [19]:
print("New shape:", df.shape)

print("\nRemote:")
print(df["is_remote"].value_counts())

print("\nHybrid:")
print(df["is_hybrid"].value_counts())

New shape: (17880, 47)

Remote:
is_remote
0    17859
1       21
Name: count, dtype: int64

Hybrid:
is_hybrid
0    17880
Name: count, dtype: int64


In [20]:
print(
    df.loc[df["is_remote"] == 1, "location"]
    .value_counts()
)

location
US, , Remote                                    2
FR, , Work from home                            2
GB, , Glasgow / Birmingham / London / Remote    1
NO, , Work from home                            1
IN, , Hyderabad/Pune/remote                     1
GR, , Remote                                    1
IT, , Work from home                            1
DK, , Work from home                            1
NL, , Work from home                            1
CH, , Work from home                            1
FI, , Work from home                            1
GB, , Work from home                            1
AU, , Work from home                            1
LU, , Work from home                            1
IE, , Work from home                            1
ES, , Work from home                            1
BE, , Work from home                            1
US, TX, Austin (or remote)                      1
PL, , REMOTE                                    1
Name: count, dtype: int64


In [21]:
print(
    df.loc[df["is_hybrid"] == 1, "location"]
)

Series([], Name: location, dtype: object)


In [22]:
# Create binary job-posting indicators

df["is_telecommuting"] = df["telecommuting"].fillna(0).astype(int)
df["has_company_logo_flag"] = df["has_company_logo"].fillna(0).astype(int)
df["has_questions_flag"] = df["has_questions"].fillna(0).astype(int)

basic_binary_features = [
    "is_telecommuting",
    "has_company_logo_flag",
    "has_questions_flag"
]

print(df[basic_binary_features].head(10))

print("\nValue counts:")
for feature in basic_binary_features:
    print(f"\n{feature}")
    print(df[feature].value_counts())

   is_telecommuting  has_company_logo_flag  has_questions_flag
0                 0                      1                   0
1                 0                      1                   0
2                 0                      1                   0
3                 0                      1                   0
4                 0                      1                   1
5                 0                      0                   0
6                 0                      1                   1
7                 0                      1                   1
8                 0                      1                   1
9                 0                      1                   0

Value counts:

is_telecommuting
is_telecommuting
0    17113
1      767
Name: count, dtype: int64

has_company_logo_flag
has_company_logo_flag
1    14220
0     3660
Name: count, dtype: int64

has_questions_flag
has_questions_flag
0    9088
1    8792
Name: count, dtype: int64


In [23]:
print(df.shape)

(17880, 50)


In [24]:
import sys
sys.path.append("../src")

from nlp.skill_dictionary import SKILL_ALIASES

In [25]:
def get_skill_features(text):
    if pd.isna(text):
        text = ""

    skills = extract_skills(str(text), SKILL_ALIASES)

    if skills is None:
        skills = []

    skill_count = len(skills)

    technical_keywords = {
        "Python",
        "C",
        "C++",
        "C#",
        "Java",
        "JavaScript",
        "SQL",
        "Machine Learning",
        "Deep Learning",
        "Artificial Intelligence",
        "Data Science",
        "Pandas",
        "NumPy",
        "Scikit-learn",
        "TensorFlow",
        "PyTorch",
        "FastAPI",
        "Docker",
        "Git"
    }

    has_technical_skills = int(
        any(skill in technical_keywords for skill in skills)
    )

    has_python = int("Python" in skills)
    has_ml_skill = int(
        any(skill in skills for skill in [
            "Machine Learning",
            "Deep Learning",
            "Artificial Intelligence"
        ])
    )

    has_data_skill = int(
        any(skill in skills for skill in [
            "Data Science",
            "Pandas",
            "NumPy",
            "SQL"
        ])
    )

    return (
        skill_count,
        has_technical_skills,
        has_python,
        has_ml_skill,
        has_data_skill
    )

In [26]:
skill_text = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["requirements"].fillna("")
)

In [27]:
import re

def extract_skills(text, skill_aliases=SKILL_ALIASES):
    found_skills = []

    for canonical_skill, aliases in skill_aliases.items():
        for alias in aliases:
            pattern = r"(?<!\w)" + re.escape(alias) + r"(?!\w)"

            if re.search(pattern, text, flags=re.IGNORECASE):
                found_skills.append(canonical_skill)
                break

    return found_skills if found_skills else None

In [29]:
# Create one combined regex for all skill aliases

all_aliases = []

for aliases in SKILL_ALIASES.values():
    all_aliases.extend(aliases)

# Remove duplicates and sort longest first
all_aliases = sorted(set(all_aliases), key=len, reverse=True)

skill_pattern = re.compile(
    r"(?<!\w)(" +
    "|".join(re.escape(alias) for alias in all_aliases) +
    r")(?!\w)",
    flags=re.IGNORECASE
)

print("Total aliases:", len(all_aliases))

Total aliases: 829


In [30]:
# Map aliases back to canonical skill names

alias_to_skill = {}

for canonical_skill, aliases in SKILL_ALIASES.items():
    for alias in aliases:
        alias_to_skill[alias.lower()] = canonical_skill


def fast_extract_skills(text):
    if pd.isna(text):
        return []

    matches = skill_pattern.findall(str(text))

    skills = set()

    for match in matches:
        skill = alias_to_skill.get(match.lower())

        if skill:
            skills.add(skill)

    return list(skills)

In [31]:
test_text = """
Looking for a Python Developer with experience in
Machine Learning, SQL, Pandas, FastAPI and Docker.
"""

print(fast_extract_skills(test_text))

['Docker', 'Python', 'Machine Learning', 'FastAPI', 'Pandas', 'SQL']


In [32]:
def get_fast_skill_features(text):
    skills = fast_extract_skills(text)

    skill_count = len(skills)

    technical_keywords = {
        "Python", "C", "C++", "C#", "Java", "JavaScript",
        "SQL", "Machine Learning", "Deep Learning",
        "Artificial Intelligence", "Data Science",
        "Pandas", "NumPy", "Scikit-learn",
        "TensorFlow", "PyTorch", "FastAPI", "Docker", "Git"
    }

    has_technical_skills = int(
        any(skill in technical_keywords for skill in skills)
    )

    has_python = int("Python" in skills)

    has_ml_skill = int(
        any(skill in skills for skill in [
            "Machine Learning",
            "Deep Learning",
            "Artificial Intelligence"
        ])
    )

    has_data_skill = int(
        any(skill in skills for skill in [
            "Data Science", "Pandas", "NumPy", "SQL"
        ])
    )

    return (
        skill_count,
        has_technical_skills,
        has_python,
        has_ml_skill,
        has_data_skill
    )

In [33]:
skill_features = skill_text.apply(get_fast_skill_features)

df["skill_count"] = skill_features.apply(lambda x: x[0])
df["has_technical_skills"] = skill_features.apply(lambda x: x[1])
df["has_python"] = skill_features.apply(lambda x: x[2])
df["has_ml_skill"] = skill_features.apply(lambda x: x[3])
df["has_data_skill"] = skill_features.apply(lambda x: x[4])

In [34]:
print("New shape:", df.shape)

print(df[
    [
        "skill_count",
        "has_technical_skills",
        "has_python",
        "has_ml_skill",
        "has_data_skill"
    ]
].describe())

New shape: (17880, 55)
        skill_count  has_technical_skills    has_python  has_ml_skill  \
count  17880.000000          17880.000000  17880.000000  17880.000000   
mean       4.039206              0.185347      0.032103      0.008277   
std        3.677073              0.388589      0.176278      0.090605   
min        0.000000              0.000000      0.000000      0.000000   
25%        1.000000              0.000000      0.000000      0.000000   
50%        3.000000              0.000000      0.000000      0.000000   
75%        6.000000              0.000000      0.000000      0.000000   
max       27.000000              1.000000      1.000000      1.000000   

       has_data_skill  
count    17880.000000  
mean         0.054195  
std          0.226408  
min          0.000000  
25%          0.000000  
50%          0.000000  
75%          0.000000  
max          1.000000  


In [35]:
def parse_experience(value):
    if pd.isna(value):
        return np.nan, np.nan, 0

    text = str(value).lower().strip()

    # Freshers / no experience
    if (
        "fresh" in text
        or "no prior experience" in text
        or "experience is not required" in text
    ):
        return 0.0, 0.0, 1

    # X-Y years
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:-|to)\s*(\d+(?:\.\d+)?)\s*years?",
        text
    )

    if match:
        return (
            float(match.group(1)),
            float(match.group(2)),
            0
        )

    # X+ years / minimum X years / at least X years
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*\+?\s*years?",
        text
    )

    if match:
        minimum = float(match.group(1))
        return minimum, np.nan, 0

    # Months
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*months?",
        text
    )

    if match:
        months = float(match.group(1))
        years = months / 12

        return years, years, 0

    return np.nan, np.nan, 0

In [36]:
experience_features = df["required_experience"].apply(parse_experience)

df["experience_min_years"] = experience_features.apply(lambda x: x[0])
df["experience_max_years"] = experience_features.apply(lambda x: x[1])
df["is_fresher_friendly"] = experience_features.apply(lambda x: x[2])

In [37]:
print("New shape:", df.shape)

print(
    df[
        [
            "required_experience",
            "experience_min_years",
            "experience_max_years",
            "is_fresher_friendly"
        ]
    ].head(20)
)

New shape: (17880, 58)
   required_experience  experience_min_years  experience_max_years  \
0           Internship                   NaN                   NaN   
1       Not Applicable                   NaN                   NaN   
2                  NaN                   NaN                   NaN   
3     Mid-Senior level                   NaN                   NaN   
4     Mid-Senior level                   NaN                   NaN   
5                  NaN                   NaN                   NaN   
6     Mid-Senior level                   NaN                   NaN   
7                  NaN                   NaN                   NaN   
8            Associate                   NaN                   NaN   
9          Entry level                   NaN                   NaN   
10    Mid-Senior level                   NaN                   NaN   
11                 NaN                   NaN                   NaN   
12           Associate                   NaN                   NaN 

In [38]:
# Normalize required experience
experience_text = (
    df["required_experience"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["is_entry_level"] = experience_text.eq("entry level").astype(int)

df["is_associate"] = experience_text.eq("associate").astype(int)

df["is_mid_senior"] = experience_text.eq("mid-senior level").astype(int)

df["is_executive"] = experience_text.eq("executive").astype(int)

df["is_internship_experience"] = experience_text.eq("internship").astype(int)

df["is_experience_not_applicable"] = (
    experience_text.eq("not applicable")
).astype(int)

experience_level_features = [
    "is_entry_level",
    "is_associate",
    "is_mid_senior",
    "is_executive",
    "is_internship_experience",
    "is_experience_not_applicable"
]

print(df[[
    "required_experience"
] + experience_level_features].head(20))

   required_experience  is_entry_level  is_associate  is_mid_senior  \
0           Internship               0             0              0   
1       Not Applicable               0             0              0   
2                  NaN               0             0              0   
3     Mid-Senior level               0             0              1   
4     Mid-Senior level               0             0              1   
5                  NaN               0             0              0   
6     Mid-Senior level               0             0              1   
7                  NaN               0             0              0   
8            Associate               0             1              0   
9          Entry level               1             0              0   
10    Mid-Senior level               0             0              1   
11                 NaN               0             0              0   
12           Associate               0             1              0   
13    

In [39]:
print(df[experience_level_features].sum())

is_entry_level                  2697
is_associate                    2297
is_mid_senior                   3809
is_executive                     141
is_internship_experience         381
is_experience_not_applicable    1116
dtype: int64


In [40]:
print(df.shape)

(17880, 64)


In [41]:
print(df["required_education"].value_counts(dropna=False))

required_education
NaN                                  8105
Bachelor's Degree                    5145
High School or equivalent            2080
Unspecified                          1397
Master's Degree                       416
Associate Degree                      274
Certification                         170
Some College Coursework Completed     102
Professional                           74
Vocational                             49
Some High School Coursework            27
Doctorate                              26
Vocational - HS Diploma                 9
Vocational - Degree                     6
Name: count, dtype: int64


In [42]:
education_text = (
    df["required_education"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["has_education_requirement"] = (
    education_text.ne("")
    & ~education_text.eq("unspecified")
).astype(int)

df["is_bachelors"] = education_text.eq(
    "bachelor's degree"
).astype(int)

df["is_masters"] = education_text.eq(
    "master's degree"
).astype(int)

df["is_doctorate"] = education_text.eq(
    "doctorate"
).astype(int)

df["is_associate_degree"] = education_text.eq(
    "associate degree"
).astype(int)

df["is_high_school"] = education_text.isin([
    "high school or equivalent",
    "some high school coursework"
]).astype(int)

df["is_certification"] = education_text.eq(
    "certification"
).astype(int)

df["is_vocational"] = education_text.str.contains(
    "vocational",
    na=False
).astype(int)

In [43]:
education_features = [
    "has_education_requirement",
    "is_bachelors",
    "is_masters",
    "is_doctorate",
    "is_associate_degree",
    "is_high_school",
    "is_certification",
    "is_vocational"
]

print(df[education_features].sum())

has_education_requirement    8378
is_bachelors                 5145
is_masters                    416
is_doctorate                   26
is_associate_degree           274
is_high_school               2107
is_certification              170
is_vocational                  64
dtype: int64


In [44]:
print(df.shape)

(17880, 72)


In [45]:
print(len(df.columns))

for i, col in enumerate(df.columns):
    print(i, col)

72
0 job_id
1 title
2 location
3 department
4 salary_range
5 company_profile
6 description
7 requirements
8 benefits
9 telecommuting
10 has_company_logo
11 has_questions
12 employment_type
13 required_experience
14 required_education
15 industry
16 function
17 fraudulent
18 has_department
19 has_salary_range
20 has_company_profile
21 has_description
22 has_requirements
23 has_benefits
24 has_employment_type
25 has_required_experience
26 has_required_education
27 has_industry
28 has_function
29 title_char_length
30 title_word_count
31 company_profile_char_length
32 company_profile_word_count
33 description_char_length
34 description_word_count
35 requirements_char_length
36 requirements_word_count
37 benefits_char_length
38 benefits_word_count
39 salary_min
40 salary_max
41 salary_is_range
42 salary_midpoint
43 salary_range_width
44 has_location
45 is_remote
46 is_hybrid
47 is_telecommuting
48 has_company_logo_flag
49 has_questions_flag
50 skill_count
51 has_technical_skills
52 has_pyth

In [46]:
df.drop(
    columns=[
        "experience_min_years",
        "experience_max_years",
        "is_fresher_friendly"
    ],
    inplace=True
)

In [47]:
print(df.shape)

(17880, 69)


In [48]:
print(
    "experience_min_years" in df.columns,
    "experience_max_years" in df.columns,
    "is_fresher_friendly" in df.columns
)


False False False


In [49]:
categorical_columns = [
    "department",
    "industry",
    "function",
    "employment_type",
    "required_experience",
    "required_education"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

print("Categorical columns prepared:")
for col in categorical_columns:
    print(f"{col}: {df[col].nunique()} unique values")

Categorical columns prepared:
department: 1277 unique values
industry: 132 unique values
function: 38 unique values
employment_type: 6 unique values
required_experience: 8 unique values
required_education: 14 unique values


In [50]:
print(df[categorical_columns].head())

print("\nMissing values:")
print(df[categorical_columns].isna().sum())

  department                   industry              function employment_type  \
0  Marketing                    Unknown             Marketing           Other   
1    Success  Marketing and Advertising      Customer Service       Full-time   
2    Unknown                    Unknown               Unknown         Unknown   
3      Sales          Computer Software                 Sales       Full-time   
4    Unknown     Hospital & Health Care  Health Care Provider       Full-time   

  required_experience required_education  
0          Internship            Unknown  
1      Not Applicable            Unknown  
2             Unknown            Unknown  
3    Mid-Senior level  Bachelor's Degree  
4    Mid-Senior level  Bachelor's Degree  

Missing values:
department             0
industry               0
function               0
employment_type        0
required_experience    0
required_education     0
dtype: int64


In [51]:
print(df.shape)

(17880, 69)


In [52]:
length_thresholds = {
    "company_profile_char_length": df["company_profile_char_length"].quantile(0.25),
    "requirements_char_length": df["requirements_char_length"].quantile(0.25),
    "description_char_length": df["description_char_length"].quantile(0.25),
    "benefits_char_length": df["benefits_char_length"].quantile(0.25)
}

print("Length thresholds:")
print(length_thresholds)

Length thresholds:
{'company_profile_char_length': np.float64(138.0), 'requirements_char_length': np.float64(146.0), 'description_char_length': np.float64(607.0), 'benefits_char_length': np.float64(0.0)}


In [53]:
df["short_company_profile"] = (
    df["company_profile_char_length"] < 
    length_thresholds["company_profile_char_length"]
).astype(int)

df["short_requirements"] = (
    df["requirements_char_length"] < 
    length_thresholds["requirements_char_length"]
).astype(int)

df["short_description"] = (
    df["description_char_length"] < 
    length_thresholds["description_char_length"]
).astype(int)

df["short_benefits"] = (
    df["benefits_char_length"] < 
    length_thresholds["benefits_char_length"]
).astype(int)

In [54]:
quality_features = [
    "short_company_profile",
    "short_requirements",
    "short_description",
    "short_benefits"
]

print(df[quality_features].sum())

print("\nNew shape:", df.shape)

short_company_profile    4468
short_requirements       4451
short_description        4457
short_benefits              0
dtype: int64

New shape: (17880, 73)


In [55]:
important_presence_features = [
    "has_salary_range",
    "has_company_profile",
    "has_requirements",
    "has_benefits",
    "has_required_experience",
    "has_required_education",
    "has_industry",
    "has_function"
]

df["missing_important_info_count"] = (
    len(important_presence_features)
    - df[important_presence_features].sum(axis=1)
)

In [56]:
text_presence_features = [
    "has_company_profile",
    "has_description",
    "has_requirements",
    "has_benefits"
]

df["text_section_count"] = df[text_presence_features].sum(axis=1)

In [57]:
df["has_any_salary"] = df["has_salary_range"]

In [60]:
df["has_any_salary"] = df["has_salary_range"]

df["has_company_information"] = df["has_company_profile"]

In [61]:
df["has_any_salary"] = df["has_salary_range"]

In [62]:
risk_features = [
    "missing_important_info_count",
    "text_section_count",
    "has_any_salary",
    "has_company_information"
]

print(df[risk_features].describe())

print("\nSample:")
print(df[risk_features].head(10))

print("\nShape:", df.shape)

       missing_important_info_count  text_section_count  has_any_salary  \
count                  17880.000000        17880.000000    17880.000000   
mean                       3.061577            3.260794        0.160403   
std                        2.153451            0.857904        0.366989   
min                        0.000000            0.000000        0.000000   
25%                        1.000000            3.000000        0.000000   
50%                        2.000000            3.000000        0.000000   
75%                        5.000000            4.000000        0.000000   
max                        8.000000            4.000000        1.000000   

       has_company_information  
count             17880.000000  
mean                  0.814989  
std                   0.388317  
min                   0.000000  
25%                   1.000000  
50%                   1.000000  
75%                   1.000000  
max                   1.000000  

Sample:
   missing_importa

In [63]:
print(
    df[
        [
            "missing_important_info_count",
            "text_section_count",
            "has_any_salary",
            "has_company_information"
        ]
    ].isna().sum()
)

missing_important_info_count    0
text_section_count              0
has_any_salary                  0
has_company_information         0
dtype: int64


In [64]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nAll feature columns:")
for i, col in enumerate(df.columns):
    print(i, col)

Total rows: 17880
Total columns: 77

All feature columns:
0 job_id
1 title
2 location
3 department
4 salary_range
5 company_profile
6 description
7 requirements
8 benefits
9 telecommuting
10 has_company_logo
11 has_questions
12 employment_type
13 required_experience
14 required_education
15 industry
16 function
17 fraudulent
18 has_department
19 has_salary_range
20 has_company_profile
21 has_description
22 has_requirements
23 has_benefits
24 has_employment_type
25 has_required_experience
26 has_required_education
27 has_industry
28 has_function
29 title_char_length
30 title_word_count
31 company_profile_char_length
32 company_profile_word_count
33 description_char_length
34 description_word_count
35 requirements_char_length
36 requirements_word_count
37 benefits_char_length
38 benefits_word_count
39 salary_min
40 salary_max
41 salary_is_range
42 salary_midpoint
43 salary_range_width
44 has_location
45 is_remote
46 is_hybrid
47 is_telecommuting
48 has_company_logo_flag
49 has_questions_

In [65]:
print("\nDuplicate column names:")
print(df.columns[df.columns.duplicated()].tolist())


Duplicate column names:
[]


In [66]:
engineered_features = [
    col for col in df.columns
    if col not in [
        "job_id",
        "title",
        "location",
        "department",
        "salary_range",
        "company_profile",
        "description",
        "requirements",
        "benefits",
        "telecommuting",
        "has_company_logo",
        "has_questions",
        "employment_type",
        "required_experience",
        "required_education",
        "industry",
        "function",
        "fraudulent"
    ]
]

print("\nEngineered feature count:", len(engineered_features))

print("\nMissing values in engineered features:")
print(
    df[engineered_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)


Engineered feature count: 59

Missing values in engineered features:
salary_range_width              15039
salary_midpoint                 15039
salary_max                      15039
salary_min                      15039
has_department                      0
has_education_requirement           0
has_python                          0
has_ml_skill                        0
has_data_skill                      0
is_entry_level                      0
is_associate                        0
is_mid_senior                       0
is_executive                        0
is_internship_experience            0
is_experience_not_applicable        0
is_bachelors                        0
skill_count                         0
is_masters                          0
is_doctorate                        0
is_associate_degree                 0
dtype: int64


In [67]:
print("Total columns:", len(df.columns))
print(df.columns.tolist())

Total columns: 77
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent', 'has_department', 'has_salary_range', 'has_company_profile', 'has_description', 'has_requirements', 'has_benefits', 'has_employment_type', 'has_required_experience', 'has_required_education', 'has_industry', 'has_function', 'title_char_length', 'title_word_count', 'company_profile_char_length', 'company_profile_word_count', 'description_char_length', 'description_word_count', 'requirements_char_length', 'requirements_word_count', 'benefits_char_length', 'benefits_word_count', 'salary_min', 'salary_max', 'salary_is_range', 'salary_midpoint', 'salary_range_width', 'has_location', 'is_remote', 'is_hybrid', 'is_telecommuting', 'has_company_logo_flag', 'has_questions_flag', 'skill_count', 'has_tech